# 06 — Standardizasyon (Hafta 8)

7 bağımsız değişken üzerinde:
1. **Skewness analizi** — hangileri sağa skewed?
2. **Log1p dönüşümü** — `|skew| > 1.0` olanlara (ve tüm değerleri ≥ 0 olanlara — NDVI hariç).
3. **Z-skor** — hepsine, log uygulananlarda log'lu kolondan, diğerlerinde orijinalden.

**Önkoşul:** `grid_30m_full.gpkg` (Hafta 6).

**Çıktı:** `grid_30m_standardized.gpkg` — orijinal + `_log` (gerekenler için) + `_z` kolonları. Hafta 9 VIF + Hafta 10 Random Forest için hazır.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    FIGURES, TABLES, GRID_30M_FULL, GRID_30M_STANDARDIZED,
    FEATURE_COLUMNS, TARGET_COLUMN, SKEW_THRESHOLD_LOG,
)
from src.standardization import (
    compute_skewness, select_log_columns, apply_standardization,
)

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# --- Yükle ---
grid = gpd.read_file(GRID_30M_FULL)
print(f"Grid: {len(grid):,} hücre")
print(f"Bağımsız değişkenler ({len(FEATURE_COLUMNS)}):")
for c in FEATURE_COLUMNS:
    print(f"  {c}")
print(f"\nHedef: {TARGET_COLUMN}")

In [ ]:
# --- Raw skewness ---
skew = compute_skewness(grid, FEATURE_COLUMNS + [TARGET_COLUMN])
print("Raw skewness (azalan |skew|):")
print(skew.round(3))

log_cols = select_log_columns(grid, FEATURE_COLUMNS, skew_threshold=SKEW_THRESHOLD_LOG)
print(f"\nLog1p uygulanacak ({len(log_cols)} kolon):")
for c in log_cols:
    s = grid[c].skew()
    print(f"  {c:30s}  skew={s:+.2f}")

In [ ]:
# --- Standardizasyon ---
grid_std, info = apply_standardization(
    grid, cols=FEATURE_COLUMNS,
    skew_threshold=SKEW_THRESHOLD_LOG,
)

z_cols = [f"{c}_z" for c in FEATURE_COLUMNS]
print("Z-skor sonrası özet (mean=0, std=1 hedef):")
print(grid_std[z_cols].agg(["mean", "std", "min", "max"]).round(3))

print("\nLog uygulanan kolonlar:")
for c, p in info["zscore_params"].items():
    src = p["source"]
    flag = "✓ log" if src.endswith("_log") else "  raw"
    print(f"  {flag} | {c:30s} mean={p['mean']:>10.3f}  std={p['std']:>10.3f}")

In [ ]:
# --- Histogram karşılaştırması: raw vs log vs z ---
n_feats = len(FEATURE_COLUMNS)
fig, axes = plt.subplots(n_feats, 3, figsize=(15, 3 * n_feats))
for i, c in enumerate(FEATURE_COLUMNS):
    has_log = c in info["log_cols"]

    # Sol — raw
    axes[i, 0].hist(grid_std[c].dropna(), bins=50, color="#264653", edgecolor="black", alpha=0.85)
    axes[i, 0].set_title(f"{c}  (raw, skew={grid_std[c].skew():.2f})", fontsize=9)

    # Orta — log (eğer uygulandıysa)
    if has_log:
        log_col = f"{c}_log"
        axes[i, 1].hist(grid_std[log_col].dropna(), bins=50, color="#E76F51", edgecolor="black", alpha=0.85)
        axes[i, 1].set_title(f"{log_col}  (skew={grid_std[log_col].skew():.2f})", fontsize=9)
    else:
        axes[i, 1].text(0.5, 0.5, "log uygulanmadı\n(skew düşük)",
                        ha="center", va="center", transform=axes[i, 1].transAxes,
                        fontsize=10, color="gray")
        axes[i, 1].axis("off")

    # Sağ — z-skor
    z_col = f"{c}_z"
    axes[i, 2].hist(grid_std[z_col].dropna(), bins=50, color="#2A9D8F", edgecolor="black", alpha=0.85)
    axes[i, 2].set_title(f"{z_col}  (mean={grid_std[z_col].mean():.2f}, std={grid_std[z_col].std():.2f})", fontsize=9)
    axes[i, 2].axvline(0, color="black", linestyle="--", linewidth=0.8)

plt.tight_layout()
plt.savefig(FIGURES / "06_standardization_histograms.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Standardize edilmiş hali ile korelasyon (Pearson değişmemeli — kontrol) ---
# log + z dönüşümü monoton, bu nedenle Spearman değişmez. Pearson da değişmemeli
# (z-skor lineer; log monoton ama lineer değil → Pearson hafif değişir).
from scipy.stats import spearmanr

raw_pearson  = grid_std[FEATURE_COLUMNS + [TARGET_COLUMN]].corr().round(3)
z_pearson    = grid_std[z_cols + [TARGET_COLUMN]].corr().round(3)
spearman_raw = grid_std[FEATURE_COLUMNS + [TARGET_COLUMN]].corr(method="spearman").round(3)

comp = pd.DataFrame({
    "raw_pearson_with_LST":  raw_pearson[TARGET_COLUMN].drop(TARGET_COLUMN).values,
    "z_pearson_with_LST":    z_pearson[TARGET_COLUMN].drop(TARGET_COLUMN).values,
    "spearman_with_LST":     spearman_raw[TARGET_COLUMN].drop(FEATURE_COLUMNS).values
        if False else spearman_raw[TARGET_COLUMN].drop(TARGET_COLUMN).values,
}, index=FEATURE_COLUMNS).round(3)

print("LST ile korelasyon karşılaştırması:")
print(comp.to_string())
print("\nSpearman raw'dan farklıysa monotonik ama lineer-olmayan ilişki vardır.")

In [ ]:
# --- Z-skor korelasyon ısı haritası ---
fig, ax = plt.subplots(figsize=(9, 7))
z_corr = grid_std[z_cols + [TARGET_COLUMN]].corr().round(3)
sns.heatmap(z_corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, cbar_kws={"label": "Pearson r"}, vmin=-1, vmax=1,
            annot_kws={"size": 9})
ax.set_title("Standardize edilmiş değişkenler korelasyon matrisi")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "06_standardization_corr.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Kaydet + summary ---
grid_std.to_file(GRID_30M_STANDARDIZED, driver="GPKG")
print(f"Kaydedildi: {GRID_30M_STANDARDIZED} ({GRID_30M_STANDARDIZED.stat().st_size/1024/1024:.2f} MB)")

summary_rows = []
for c in FEATURE_COLUMNS:
    p = info["zscore_params"][c]
    summary_rows.append({
        "degisken": c,
        "raw_skew": round(float(grid_std[c].skew()), 3),
        "log_uygulandi": c in info["log_cols"],
        "z_kaynak": p["source"],
        "z_mean": round(p["mean"], 3),
        "z_std": round(p["std"], 3),
        "r_with_LST_raw":  round(float(grid_std[[c, TARGET_COLUMN]].corr().iloc[0, 1]), 3),
        "r_with_LST_z":    round(float(grid_std[[f"{c}_z", TARGET_COLUMN]].corr().iloc[0, 1]), 3),
        "r_with_LST_spearman": round(float(grid_std[[c, TARGET_COLUMN]].corr(method="spearman").iloc[0, 1]), 3),
    })
summary = pd.DataFrame(summary_rows)
summary.to_csv(TABLES / "06_standardization_summary.csv", index=False, encoding="utf-8-sig")
print("\nÖzet:")
print(summary.to_string(index=False))

## Özet

- 7 değişken üzerinden skewness ölçüldü; `|skew| > 1.0` ve tüm değerler ≥ 0 olanlara log1p uygulandı.
- Z-skor (mean=0, std=1) tüm değişkenlere uygulandı (log uygulanansa log'lu kolondan).
- Sonuç tablosu: orijinal + `_log` (gerekenler için) + `_z` kolonları.
- Pearson vs Spearman karşılaştırması: monotonik ama lineer-olmayan ilişkileri ortaya çıkardı (ileride RF feature importance ile uyumlu yorumlama için).

## Sonraki Adım

`07_multicollinearity.ipynb` (Hafta 9) — VIF analizi: 7 değişkenden multicollinear olanları (VIF > 10) eleyip son 5 değişkenli modelleme setini seçeceğiz.